# CircuitKIT: Discovering Circuits on Your Own Custom Text

This notebook demonstrates how to:
1. Define your own custom text prompts (clean vs counterfactual pairs).
2. Dynamically register a custom task in CircuitKIT.
3. Run **Circuit Discovery** (EAP-IG) on `gpt2`.
4. Inspect top critical Attention Heads and MLP layers.
5. Evaluate **Multi-Pillar Faithfulness** (Causal Patching & Ablation).
6. Structurally **prune** the model down to the circuit.
7. Render the **interactive D3.js circuit graph** inline.
8. Export the **Hugging Face checkpoint**.

In [ ]:
from pathlib import Path
import pandas as pd
import yaml
import torch

# Always import circuitkit first
import circuitkit as ck
from circuitkit.tasks.yaml_loader import YAMLTaskLoader
from circuitkit.tasks.registry import register_task, is_task_registered

print("CircuitKIT version:", getattr(ck, "__version__", "0.2.0"))

## 1. Define Your Custom Prompts
A task needs contrastive pairs to trace causality:
- `prompt`: Clean prompt where the model performs the behavior.
- `answer`: Target token (include a leading space if typical for tokenizers, e.g. `' Paris'`).
- `corrupted_prompt`: Counterfactual prompt with key entity modified.
- `corrupted_answer`: Alternate/distractor token.

In [ ]:
MY_PROMPTS = [
    {
        "prompt": "The capital of France is",
        "answer": " Paris",
        "corrupted_prompt": "The capital of Italy is",
        "corrupted_answer": " Rome"
    },
    {
        "prompt": "The capital of Spain is",
        "answer": " Madrid",
        "corrupted_prompt": "The capital of Germany is",
        "corrupted_answer": " Berlin"
    },
    {
        "prompt": "The capital of Japan is",
        "answer": " Tokyo",
        "corrupted_prompt": "The capital of China is",
        "corrupted_answer": " Beijing"
    },
    {
        "prompt": "The capital of England is",
        "answer": " London",
        "corrupted_prompt": "The capital of Egypt is",
        "corrupted_answer": " Cairo"
    }
]

# Convert to DataFrame and preview
df_prompts = pd.DataFrame(MY_PROMPTS)
df_prompts

## 2. Register Custom Task in CircuitKIT

In [ ]:
TASK_NAME = "my_notebook_task"
task_dir = Path("./custom_tasks") / TASK_NAME
task_dir.mkdir(parents=True, exist_ok=True)

csv_path = task_dir / "data.csv"
yaml_path = task_dir / "task.yaml"

# Save CSV
df_prompts.to_csv(csv_path, index=False)

# Create YAML specification
task_config = {
    "name": TASK_NAME,
    "source": {"type": "csv", "path": str(csv_path)},
    "schema": {
        "prompt": "prompt",
        "answer": "answer",
        "corrupted_prompt": "corrupted_prompt",
        "corrupted_answer": "corrupted_answer"
    },
    "metric": "logit_diff"
}

with open(yaml_path, "w") as f:
    yaml.dump(task_config, f, default_flow_style=False)

# Register dynamically (overwrite=True allows safe cell re-runs)
spec = YAMLTaskLoader.load(yaml_path)
register_task(spec, overwrite=True)
print(f"✓ Task '{TASK_NAME}' registered successfully! Is active:", is_task_registered(TASK_NAME))

## 3. Load Model and Discover the Circuit

In [ ]:
# Load GPT-2 into HookedTransformer
model = ck.load_model("gpt2")

# Run EAP-IG Discovery
circuit = ck.discover(
    model,
    task=TASK_NAME,
    algorithm="eap-ig",
    level="node",
    sparsity=0.2,       # Retain top 80% critical nodes
    n_examples=len(MY_PROMPTS),
    batch_size=2,
    ig_steps=2,
    output_path="./results_custom/circuit.pt"
)

print(f"Discovered circuit containing {len(circuit)} nodes.")

## 4. Inspect Top Critical Components

In [ ]:
top_nodes = circuit.top_nodes(10)
for comp, score in top_nodes.items():
    print(f"{comp:10s} : {score:.6f}")

## 5. Evaluate Multi-Pillar Faithfulness

In [ ]:
report = ck.faithfulness(
    model,
    circuit,
    task=TASK_NAME,
    pillars=["patching", "ablation"],
    n_examples=len(MY_PROMPTS)
)

print(f"Pillar 1 (Causal Patching Score): {report.patching_score:.4f}")
print(f"Pillar 2 (Ablation Drop Score):   {report.ablation_score:.4f}")

## 6. Render the Interactive D3 Graph in Notebook

In [ ]:
from IPython.display import HTML

viz_path = "./results_custom/circuit_graph.html"
ck.visualize_circuit(circuit, mode="graph", output=viz_path)

with open(viz_path, "r") as f:
    html_content = f.read()

HTML(f'<iframe src="{viz_path}" width="100%" height="600" style="border:none;"></iframe>')

## 7. Structurally Prune & Export Hugging Face Checkpoint

In [ ]:
# Prune weights outside the circuit
pruned_model = ck.prune(model, circuit, sparsity=0.2, scope="both")

# Export standard Hugging Face checkpoint
ckpt_dir = "./results_custom/checkpoint"
saved_path = ck.export_checkpoint(pruned_model, circuit, ckpt_dir)
print("Checkpoint written to:", saved_path)